# Le jour où M1 a triplé — sans un dollar de plus · *The day M1 tripled — without a single new dollar*

Notebook compagnon du chapitre **25. Masse monétaire M1, M2 : ce que ces agrégats mesurent vraiment** — [lire l'article](https://nmlab.io/ressources/masse-monetaire-m1-m2).
Companion notebook to chapter **25. Money Supply M1, M2: What These Aggregates Really Measure** — [read the article](https://nmlab.io/en/ressources/money-supply-m1-m2).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# (les séries sont chargées dans build_figure)


import numpy as np
import pandas as pd
from matplotlib.figure import Figure
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

C, W = nm.COLORS, nm.WIDTH_PX

EA_M3_KEY = "BSI/M.U2.Y.V.M30.X.1.U2.2300.Z01.E"   # encours M3 zone euro (BCE)


def load(series_id: str, start: str | None = None, end: str | None = None) -> pd.Series:
    """Charge une série en direct : FRED, ou le portail de la BCE pour « EA_M3 »."""
    if series_id == "EA_M3":
        url = (f"https://data-api.ecb.europa.eu/service/data/{EA_M3_KEY}"
               "?format=csvdata&detail=dataonly")
        raw = pd.read_csv(url)
        s = pd.Series(raw["OBS_VALUE"].values,
                      index=pd.PeriodIndex(raw["TIME_PERIOD"], freq="M").to_timestamp())
        s = s.sort_index() / 1000.0                 # millions -> milliards d'euros
    else:
        s = nm.load_fred(series_id)
    return s.loc[start:end]


def T(d: dict, lang: str):
    """Sélectionne le jeu de libellés de la langue demandée."""
    return d[lang]


def build_figure(lang: str = "fr") -> Figure:
    """Construit la figure NMLab du chapitre (libellés selon ``lang``)."""
    m1 = load("M1SL","2015-01")/1000.0
    fig = nm.figure(1045); ax = nm.axes(fig)
    ax.plot(m1.index, m1.values, color=C["blue"], lw=3.2, solid_capstyle="round")
    x = pd.Timestamp("2020-05-01")
    ax.annotate("", xy=(x,16.3), xytext=(x,4.9),
                arrowprops=dict(arrowstyle="-|>", color=C["rose"], lw=2.6, shrinkA=0, shrinkB=0))
    d = dict(fr=("Le jour où M1 a triplé — sans un dollar de plus",
                 "Masse monétaire M1 des États-Unis, en billions de dollars.",
                 "avril 2020 : 4,9","mai 2020 : 16,3",
                 "En mai 2020, les comptes d'épargne sont reclassés dans M1 (assouplissement de la Regulation D) :\nune rupture de définition, pas de la monnaie neuve. Source : FRED (M1SL)."),
             en=("The month M1 tripled — without one extra dollar",
                 "United States M1 money stock, in trillions of dollars.",
                 "Apr. 2020: 4.9","May 2020: 16.3",
                 "In May 2020 savings accounts were reclassified into M1 (Regulation D easing):\na break in definition, not new money. Source: FRED (M1SL)."))
    t = T(d,lang); nm.header(fig,t[0],t[1])
    ax.text(pd.Timestamp("2019-01-01"),5.6,t[2],color=C["muted"],fontsize=19,ha="right",va="center")
    ax.text(pd.Timestamp("2020-08-15"),15.4,t[3],color=C["rose"],fontsize=19.5,ha="left",va="center",fontweight="bold")
    ax.set_ylim(0,22); ax.set_xlim(pd.Timestamp("2015-01-01"),pd.Timestamp("2026-08-01"))
    nm.footer(fig,t[4]);
    return fig


build_figure(LANG)